# Salary Prediction

Corresponding  with this notebook is a slide deck where you will need to update all the portions in red.  Completing the notebook will provide all the results needed for the slides.  **Correctly completing the slides is a required part of the project.**


## Table of Contents
- [Introduction](#intro)
- [Part I - Descriptive Statistics](#descriptive)
- [Part II - Regression](#regression)
- [Part III - Interpret Results](#interpret)


<a id='intro'></a>
### Introduction

Linear Regression is very commonly performed by data analysts and data scientists.  For this project, you will be working to understand the results of a Linear Regression model associated with salaries.  Your goal is to work through this notebook to understand what variables are related to salary, and how exactly they are related.

As a final check, assure you meet all the criteria on the rubric.

<a id='descriptive'></a>
#### Part I - Descriptive Statistics

To get started, let's import our libraries.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
random.seed(0)

For each of the parts of question `1` notice links to [pandas documentation](https://pandas.pydata.org/) is provided to assist with answering the questions.  Though there are other ways you could solve the questions, the documentation is provided to assist you with one fast way to find the answer to each question.


`1.a)` Now, read in the `salary_data.csv` data. Store it in `df`. Read in the dataset and take a look at the top few rows here. **This question is completed for you**:

In [2]:
df = pd.read_csv('salary_data.csv')
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


`b)` Use the below cell to find the number of rows in the dataset. [Helpful pandas link - `Dataframe.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html#pandas.DataFrame.shape)

In [ ]:
df.shape[0]

`c)` Do any of the rows have missing values? [Helpful Pandas Link One](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html) and [Helpful Pandas Link Two](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)  

If there are missing values, determine a method for dealing with them.

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# There are 2 missing values in each column. Since this is a small number (< 1%),
# we will drop these rows.
df = df.dropna()
print(f"After dropping missing values: {df.shape[0]} rows remain")

`d)` How many employees are in each `Education Level`? Build a bar chart to show the count of employees in each level.

In [ ]:
df['Education Level'].value_counts()

In [ ]:
# bar chart of results - this part is done for you
df['Education Level'].value_counts().plot(kind='bar');
plt.title('Number of Visits From Each Education Level');
plt.ylabel('Count of Visits');
plt.show();

`e)` What are the possible values for `Salary`?  What does the distribution of `Salary` look like?

In [ ]:
# Salary statistics
print(f"Salary Range: ${df['Salary'].min():,.0f} - ${df['Salary'].max():,.0f}")
print(f"Mean Salary: ${df['Salary'].mean():,.2f}")
print(f"Median Salary: ${df['Salary'].median():,.2f}")
print(f"Std Dev: ${df['Salary'].std():,.2f}")

In [ ]:
# Salary distribution
df['Salary'].hist(bins=20)
plt.title('Distribution of Salary')
plt.xlabel('Salary ($)')
plt.ylabel('Frequency')
plt.show()

<a id='regression'></a>
#### Part II - Regression

`1.` Now that you have had a chance to learn more about the dataset, let's look more at how different factors are related to `Salary`.

`a)` Consider average salary by gender, is there evidence that salaries are higher for one gender over the other?

In [ ]:
# Average salary by gender
df.groupby('Gender')['Salary'].mean()
# Males earn ~$103,868 vs Females ~$97,011 — a ~$6,800 difference suggesting
# gender may be associated with salary.

`b)` Consider average salary by education level, is there evidence that salaries are higher for increased education?

In [ ]:
# Average salary by education level
df.groupby('Education Level')['Salary'].mean()
# Clear positive relationship: Bachelor's ~$74,756 < Master's ~$129,796 < PhD ~$157,843

`c)` Consider average salary by years of experience, is there evidence that salaries are associated with increased years of experience?

In [ ]:
# Scatter plot: Years of Experience vs Salary
plt.scatter(df['Years of Experience'], df['Salary'], alpha=0.5)
plt.title('Years of Experience vs Salary')
plt.xlabel('Years of Experience')
plt.ylabel('Salary ($)')
plt.show()

# Correlation
print(f"Correlation: {df['Years of Experience'].corr(df['Salary']):.4f}")
# Strong positive correlation (~0.92) — salary increases with experience.

`d)`  To make use of Job Title column lets create bool flag based on word existiance


List of words:

* Director
* Junior
* Senior
* Manager
* Analyst
* Engineer

**This question is completed for you**

In [ ]:
flag_words = ['director', 'junior', 'senior', 'manager', 'analyst', 'engineer']
df['Job Title'] = df['Job Title'].str.lower()

for word in flag_words:
    df['is_' + word] = df['Job Title'].str.contains(word)
    
df = df.drop('Job Title', axis=1)

`e)` Create a flag for gender where 1 is if a person is male and 0 if the person is not.

In [ ]:
# Create gender flag: 1 = Male, 0 = Female
df['is_male'] = (df['Gender'] == 'Male').astype(int)

# Create education dummies (Bachelor's is the baseline)
df['has_masters'] = (df['Education Level'] == "Master's").astype(int)
df['has_phd'] = (df['Education Level'] == 'PhD').astype(int)

df[['is_male', 'has_masters', 'has_phd']].head()

`f)` Use statsmodels to fit a linear model to predict salary using each of the features from `a-e`.  These include:
* Gender
* Job TItle
* Years of Experience
* Education

In [ ]:
import statsmodels.api as sm

# Define features
features = ['Years of Experience', 'is_male', 'has_masters', 'has_phd',
            'is_director', 'is_junior', 'is_senior', 'is_manager', 'is_analyst', 'is_engineer']

X = sm.add_constant(df[features].astype(float))
y = df['Salary']

# Fit OLS model
model_full = sm.OLS(y, X).fit()
model_full.summary()

<a id='interpretation'></a>
### Part III - Interpret Results

`1.` Consider you are tasked with finding which features in your linear model are significantly related to salary.  Were there any features that were not significantly related to salary in your first model?  If not, remove those features and fit a new model.  Only keep the features that were significant from the original model.

In [ ]:
# From the full model, three features are NOT significant (p > 0.05):
# - is_junior (p = 0.068)
# - is_analyst (p = 0.933)
# - is_engineer (p = 0.508)
# Remove these and fit a reduced model.

sig_features = ['Years of Experience', 'is_male', 'has_masters', 'has_phd',
                'is_director', 'is_senior', 'is_manager']

X_reduced = sm.add_constant(df[sig_features].astype(float))
model_reduced = sm.OLS(y, X_reduced).fit()
model_reduced.summary()

`a)` With each additional year of experience, what is the expected change in salary?  What is the 95% confidence interval of the change?

**Years of Experience coefficient: $5,104.44**

With each additional year of experience, salary is expected to increase by approximately **$5,104**.

The 95% confidence interval is **($4,755.93, $5,452.95)**.

We can be 95% confident that each additional year of experience is associated with a salary increase between $4,756 and $5,453, holding all other variables constant.

`b)` What is the expected difference in salary between someone with a senior title and someone with none of the other title indications?

**is_senior coefficient: $14,446.48**

Employees with a senior title are expected to earn approximately **$14,446 more** than employees without any of the title indicators (director, senior, manager), holding all other variables constant.

The 95% confidence interval is **($10,867, $18,026)**. We can be 95% confident that having a senior title is associated with a salary increase between $10,867 and $18,026.

`c)` What is the expected difference in salary between someone with a PhD and an individual with no PhD nor master's degree?  What is the 95% confidence interval of the change?

**has_phd coefficient: $23,080.85**

Employees with a PhD are expected to earn approximately **$23,081 more** than employees with only a Bachelor's degree (the baseline), holding all other variables constant.

The 95% confidence interval is **($17,652, $28,510)**. We can be 95% confident that having a PhD is associated with a salary increase between $17,652 and $28,510 compared to a Bachelor's degree holder.

`d)` If a male employee has 5 years of experience as a senior engineer with a bachelor's degree, what is the expected salary of the employee?

**Predicted Salary = $28,760.78 + 5 × $5,104.44 + 1 × $7,629.65 + 1 × $14,446.48 = $76,359.11**

Note: `is_engineer` was removed from the reduced model (not statistically significant), so it does not contribute to the prediction. The prediction uses: intercept + Years of Experience (5) + is_male (1) + is_senior (1). has_masters, has_phd, is_director, is_manager are all 0 for this employee.

The expected salary is approximately **$76,359**.

`e)` Imagine that the employee in question `d)` actually has a salary of $110,000, what would the residual be for this employee?

**Residual = Actual Salary - Predicted Salary = $110,000 - $76,359 = $33,641**

The positive residual means the employee earns $33,641 more than the model predicts. This could indicate that other factors not captured in the model (e.g., specific skills, company size, location, performance) contribute to their higher salary.

`f)` How well do you think your model fits?  What metrics or plots would you consider to understand if this model does a good job of predicting salary?

In [ ]:
# Model fit metrics
print(f"R-squared: {model_reduced.rsquared:.4f}")
print(f"Adjusted R-squared: {model_reduced.rsquared_adj:.4f}")
print(f"F-statistic: {model_reduced.fvalue:.2f}")
print(f"F p-value: {model_reduced.f_pvalue:.2e}")

# Residual plot
plt.scatter(model_reduced.fittedvalues, model_reduced.resid, alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuals vs Fitted Values')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.show()

# Residual histogram
model_reduced.resid.hist(bins=20)
plt.title('Distribution of Residuals')
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.show()

**Model Fit Assessment:**

The model fits the data well:
- **R² = 0.9145** — The model explains approximately 91.5% of the variance in salary
- **Adjusted R² = 0.9128** — Very close to R², indicating the model is not overfitting
- **F-statistic is highly significant (p ≈ 0)** — The model as a whole is statistically significant
- **Residual plot** shows residuals roughly randomly scattered around zero, suggesting the linear model assumptions are reasonably met
- **Residual histogram** appears approximately normal, supporting the normality assumption

Overall, this is a strong model that captures most of the variation in salary using years of experience, gender, education level, and job title indicators.

<a id='finalcheck'></a>
## Final Check!

Congratulations!  You have reached the end of the salary prediction project!  You should be very proud of all you have accomplished!

> **Tip**: Once you are satisfied with your work here, check over your notebook to make sure that it satisfies all the specifications mentioned in the rubric. You should also probably remove all of the "Hints" and "Tips" like this one so that the presentation is as polished as possible.


<a id='submission'></a>
## Submission

Please follow the directions in the classroom to submit this notebook, as well as your completed slides.